In [1]:
import os
import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd

from shapely.geometry import LineString, MultiLineString
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
from shapely.geometry import box
import matplotlib.colors as mcolors

# ============================
# User settings
# ============================

# Generator site list
gen_csv = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv")

# BARRA-C2 variable paths
u_path = "/g/data/ob53/BARRA2/output/reanalysis/AUST-04/BOM/ERA5/historical/hres/BARRA-C2/v1/1hr/ua100m/latest/"
v_path = "/g/data/ob53/BARRA2/output/reanalysis/AUST-04/BOM/ERA5/historical/hres/BARRA-C2/v1/1hr/va100m/latest/"


In [2]:
hw_ds = xr.open_dataset(f"/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_BARRA-C2_heatwave_composites.nc")
bs_ds = xr.open_dataset(f"/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_BARRA-C2_baseline_composites.nc")

In [3]:
ds = hw_ds

extent = [ds.coords['lon'].min().item(),
          ds.coords['lon'].max().item(),
          ds.coords['lat'].min().item(),
          ds.coords['lat'].max().item()]
lon_min, lon_max, lat_min, lat_max = extent

print(f"Currently plotting {ds.attrs['mode']} days from {ds.attrs['reanalysis']}")
print(f"Extent is \n Longitudes: {lon_min} to {lon_max} \n Latitudes: {lat_min} to {lat_max}")

Currently plotting heatwave days from BARRA-C2
Extent is 
 Longitudes: 147.5 to 150.98 
 Latitudes: -38.49 to -33.53


In [4]:
if ds.attrs['mode'] == 'heatwave' and ds.attrs['reanalysis'] == 'BARRA-C2':
    output_dir = "/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/hw/C2"
    mode_str = 'heatwave'
elif ds.attrs['mode'] == 'baseline' and ds.attrs['reanalysis'] == 'BARRA-C2':
    output_dir = "/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/no_hw/C2"
    mode_str = 'baseline'
elif ds.attrs['mode'] == 'heatwave' and ds.attrs['reanalysis'] == 'BARRA-R2':
    output_dir = "/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/hw/R2"
    mode_str = 'heatwave'
elif ds.attrs['mode'] == 'baseline' and ds.attrs['reanalysis'] == 'BARRA-R2':
    output_dir = "/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/no_hw/R2"

In [5]:
# Load shapefile
gdf = gpd.read_file('/g/data/ng72/ms5578/ID_HW_BARRA/data/raw/contours/aus25cgd_l.shp').to_crs(epsg=4326)
bbox = box(lon_min, lat_min, lon_max, lat_max)
gdf_clip = gdf[gdf.geometry.intersects(bbox)]

In [6]:
# Statistically significant w ssmin=20, and Mann-Whitney 
cluster = ['TARALGA1','CROOKWF2',
            'GULLRWF1',
            'GUNNING1',
            'CRURWF1',
            'WOODLWN1',
            'BOCORWF1',
            'BODWF1']

cluster = gen_csv[gen_csv['DUID'].isin(cluster)][['DUID','lat','lon']]

In [7]:
def make_filename(t, output_dir, prefix="wind"):
    try:
        # If t is datetime-like, use date formatting
        dt_str = np.datetime_as_string(t, unit='m')
        dt_str = dt_str.replace('-', '')[2:8] + '_' + dt_str[11:13] + dt_str[14:16]
    except Exception:
        # If it's just an index/hour, format as hour
        if isinstance(t, (int, np.integer)):
            dt_str = f"hour{t:02d}"
        else:
            dt_str = str(t).replace(":", "").replace(" ", "_")
    
    return os.path.join(output_dir, f"{prefix}_{dt_str}.png")

In [8]:
def plot_variance_frame(field, lat, lon,
                         cluster=None,
                         highlight_id='BOCORWF1',
                         shapefile_gdf=None,
                         output_dir=f"{output_dir}/variance/",
                         extent=extent,
                         vmin=None,
                         vmax=None,
                         t=None,
                         mode_str=mode_str,
                         levels=None):

    
    # Create figure
    fig, ax = plt.subplots(figsize=(10, 8),
                           subplot_kw={'projection': ccrs.PlateCarree()})
    ax.set_extent(extent, crs=ccrs.PlateCarree())

    # Base map
    ax.add_feature(cfeature.COASTLINE, lw=1.0)
    ax.add_feature(cfeature.STATES, linestyle='-', lw=1.0)
    
    
    # Filled contours
    cf = ax.contourf(lon, lat, field,
                     cmap='viridis', levels=levels, vmin=0,vmax=vmax,
                     transform=ccrs.PlateCarree(), zorder=1)
    plt.colorbar(cf, ax=ax, label=f"Windspeed variance [m/s]^2")
    
    # --- Shapefile contours ---
    if shapefile_gdf is not None and not shapefile_gdf.empty:
        for geom in shapefile_gdf.geometry:
            ax.add_geometries([geom], crs=ccrs.PlateCarree(),
                              facecolor="none", edgecolor="black",
                              linewidth=0.5, zorder=3)
    
    # Cluster points
    if cluster is not None:
        ax.scatter(cluster['lon'], cluster['lat'], color="orange", edgecolor="black", s=50,
                   label="Wind Farms", zorder=4, transform=ccrs.PlateCarree())
        if highlight_id is not None and highlight_id in cluster['DUID'].values:
            highlight_point = cluster.loc[cluster['DUID'] == highlight_id]
            ax.scatter(highlight_point['lon'], highlight_point['lat'],
                       color="red", edgecolor="black", s=60,
                       label=f"ID {highlight_id}", zorder=5, transform=ccrs.PlateCarree())
    
    plt.title(f"Variance of {mode_str} day composite at hour {t} (AEST)")
    
    # Save
    filename = make_filename(t, output_dir, prefix='variance')
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return filename


In [9]:
def plot_vector_frame(u, v, lat, lon, t, output_dir=f"{output_dir}/hourly_composites/",
                     quiver_scale=None, extent=extent, cluster=cluster, highlight_id='BOCORWF1',
                     vmin=None, vmax=None, levels=None,
                     mode_str=mode_str, step=1):
    """
    Plot wind vectors with optional cluster points.
    
    cluster: DataFrame with columns ['ID', 'lat', 'lon']
    highlight_id: specific ID to highlight in red
    """
    speed = np.sqrt(u**2 + v**2)
    plt.figure(figsize=(10,8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, lw=1.5)
    ax.add_feature(cfeature.BORDERS, linestyle='-', lw=1.5)
    ax.add_feature(cfeature.STATES, linestyle='-', lw=1.5)
    
    # Contour of wind speed
    plt.contourf(lon, lat, speed, cmap="cividis", levels=levels,
                 vmin=0, vmax=12, transform=ccrs.PlateCarree())
    plt.colorbar(label='Wind speed (m/s)')
    
    # Quiver vectors
    plt.quiver(lon[::step], lat[::step], u[::step, ::step], v[::step, ::step],
               scale=quiver_scale, color='white', transform=ccrs.PlateCarree())
    
    # Plot cluster points
    if cluster is not None:
        # All points in orange
        ax.scatter(cluster['lon'], cluster['lat'], color="orange", alpha=0.6, s=50, label="Wind Farms", zorder=5, transform=ccrs.PlateCarree())
        
        # Highlight one point in red
        if highlight_id is not None and highlight_id in cluster['DUID'].values:
            highlight_point = cluster.loc[cluster['DUID'] == highlight_id]
            ax.scatter(highlight_point['lon'], highlight_point['lat'], 
                       color="red", alpha=0.6, s=80, label=f"ID {highlight_id}", zorder=6,
                       transform=ccrs.PlateCarree())

    plt.title(f'Composite of {mode_str}-day wind vectors at hour {str(t)} (AEST)')
    
    filename = make_filename(t, output_dir)
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved: {filename}')
    return filename

In [11]:
def plot_divergence_frame(divergence, lat, lon, t,
                          cluster=None, highlight_id='BOCORWF1',
                          shapefile_gdf=None,
                          output_dir=f"{output_dir}/divergence_composite",
                          extent=None,
                          mode_str="",
                          abs_max=None):
    """
    Plot scalar divergence with optional cluster points and shapefile contours.
    Uses fixed levels and norm for consistent colorbar across frames.
    """
    fig, ax = plt.subplots(figsize=(10,8), subplot_kw={'projection': ccrs.PlateCarree()})
    if extent:
        ax.set_extent(extent, crs=ccrs.PlateCarree())

    # Base map
    ax.add_feature(cfeature.COASTLINE, lw=1.0)
    ax.add_feature(cfeature.STATES, linestyle='-', lw=1.0)

    # Divergence field
    levels = np.linspace(-abs_max, abs_max, 21)
    cf = ax.contourf(lon, lat, divergence,
                 cmap='coolwarm',
                 levels=levels,
                 extend='both',
                 transform=ccrs.PlateCarree(),
                 zorder=1)
    cbar = plt.colorbar(cf, ax=ax, label='Divergence (1/s)')

    # --- Shapefile contours ---
    if shapefile_gdf is not None and not shapefile_gdf.empty:
        for geom in shapefile_gdf.geometry:
            ax.add_geometries([geom], crs=ccrs.PlateCarree(),
                              facecolor='none', edgecolor='black',
                              linewidth=0.5, zorder=3)

    # Cluster points
    if cluster is not None:
        ax.scatter(cluster['lon'], cluster['lat'], color="orange", edgecolor='black', s=50,
                   label="Wind Farms", zorder=4, transform=ccrs.PlateCarree())
        if highlight_id is not None and highlight_id in cluster['DUID'].values:
            highlight_point = cluster.loc[cluster['DUID'] == highlight_id]
            ax.scatter(highlight_point['lon'], highlight_point['lat'],
                       color="red", edgecolor='black', s=60,
                       label=f"ID {highlight_id}", zorder=5, transform=ccrs.PlateCarree())

    plt.title(f'Divergence of {mode_str} composite wind field at hour {t} (AEST)')

    # Save
    os.makedirs(output_dir, exist_ok=True)
    filename = make_filename(t, output_dir, prefix='divergence')
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return filename

def calc_divergence(ds):
    """
    Compute divergence of u/v on a spherical Earth using xarray.
    ds should be a Dataset with dimensions lat x lon (or time x lat x lon).
    """
    R = 6371000  # Earth radius in meters
    
    # lat_rad shape (lat,)
    lat_rad = np.deg2rad(ds['lat'].values)
    
    dx_1d = np.gradient(ds['lon'].values) * (np.pi/180) * R
    dx2d = dx_1d[None, :] * np.cos(lat_rad[:, None])  # shape (lat, lon)
    
    dy2d = np.gradient(ds['lat'].values) * (np.pi/180) * R
    dy2d = dy2d[:, None]  # shape (lat, 1) to broadcast with u/v

    # Compute divergence
    divergence = (ds['u_mean'].differentiate('lon') / dx2d +
                  ds['v_mean'].differentiate('lat') / dy2d)
    
    return divergence

def get_shared_divergence_norm(ds1, ds2, var_name='divergence'):
    ds1 = calc_divergence(ds1)
    ds2 = calc_divergence(ds2)
    """
    Compute symmetric TwoSlopeNorm from two datasets to define a shared color scale.
    """
    vals1 = ds1.values.flatten()
    vals2 = ds2.values.flatten()
    vals1 = vals1[~np.isnan(vals1)]
    vals2 = vals2[~np.isnan(vals2)]

    abs_max = max(np.max(np.abs(vals1)), np.max(np.abs(vals2)))
    return abs_max

In [12]:
lat_subset = ds['lat'].values
lon_subset = ds['lon'].values

In [13]:
# Calculating shared colorbars for variance plots
global_min = min(hw_ds['windspeed_variance'].min().values, bs_ds['windspeed_variance'].min().values)
global_max = max(hw_ds['windspeed_variance'].max().values, bs_ds['windspeed_variance'].max().values)
levels = np.linspace(global_min, global_max, 21) 

for t in range(len(ds['hour'])):
    time_slice = ds['windspeed_variance'].isel(hour=t).values
    
    filestring = plot_variance_frame(time_slice, ds['lat'].values, ds['lon'].values, cluster=cluster,
                     shapefile_gdf=gdf, extent=extent, vmin=global_min, vmax=global_max, t=t, levels=levels)
    
    print(f'Saved variance plots: {filestring}')

In [14]:
# Plot composite vector fields
global_min = min(hw_ds['windspeed_mean'].min().values, bs_ds['windspeed_mean'].min().values)
global_max = max(hw_ds['windspeed_mean'].max().values, bs_ds['windspeed_mean'].max().values)
levels = np.linspace(0, global_max, 21) 

for t in range(len(ds.hour)):

    
    # --- Full resolution (for contours) ---
    u_full = ds['u_mean'].isel(hour=t).sel(
        lat=lat_subset, lon=lon_subset
    ).values
    v_full = ds['v_mean'].isel(hour=t).sel(
        lat=lat_subset, lon=lon_subset
    ).values

    target_nx, target_ny = 60, 90  # about this many arrows across lon/lat
    lon_idx = np.linspace(0, len(lon_subset)-1, target_nx, dtype=int)
    lat_idx = np.linspace(0, len(lat_subset)-1, target_ny, dtype=int)
    
    lon_quiv = lon_subset[lon_idx]
    lat_quiv = lat_subset[lat_idx]
    u_quiv = u_full[np.ix_(lat_idx, lon_idx)]
    v_quiv = v_full[np.ix_(lat_idx, lon_idx)]

    # Time

    global_min = np.min(np.sqrt(u_full**2 + v_full**2))  # u_all, v_all should be full arrays (time, lat, lon)
    global_max = np.max(np.sqrt(u_full**2 + v_full**2))

    # Call plotting function with full fields for contours
    # and stepped ones for quivers
    plot_vector_frame(
        u_quiv, v_quiv, lat_quiv, lon_quiv,  # quiver data
        t,
        vmin= 0,
        vmax=global_max,
        levels=levels,
        quiver_scale=100,
        extent=extent,
        step=2
    )

In [15]:
# Shared colour bar for diverging data
abs_max = get_shared_divergence_norm(hw_ds,bs_ds)

# Compute divergence for ds
div = calc_divergence(ds)

# Loop over hours and plot
for t in range(len(ds['hour'])):
    div_time = div.isel(hour=t).values

    filestring = plot_divergence_frame(
        div_time, lat_subset, lon_subset, t,
        cluster=cluster,
        shapefile_gdf=gdf,
        extent=extent,
        mode_str=ds.attrs.get("mode", ""),
        abs_max=abs_max   # Shared color scale
        )

    print(f"Saved divergence plot: {filestring}")

# Clean up temporary arrays
del div

Saved divergence plot: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/hw/C2/divergence_composite/divergence_hour00.png
Saved divergence plot: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/hw/C2/divergence_composite/divergence_hour01.png
Saved divergence plot: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/hw/C2/divergence_composite/divergence_hour02.png
Saved divergence plot: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/hw/C2/divergence_composite/divergence_hour03.png
Saved divergence plot: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/hw/C2/divergence_composite/divergence_hour04.png
Saved divergence plot: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/hw/C2/divergence_composite/divergence_hour05.png
Saved divergence plot: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/hw/C2/divergence_composite/divergence_hour06.png
Saved divergence plot: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/hw/C2/divergence_composit

In [ ]:
abs_max